# 최신 버전 기준 유사도 검색 통합 예제

이 노트북은 기존 3개 자료를 **하나로 합치고**, 오래된 LangChain / OpenAI / Chroma 사용법을 **최신 패키지 구조 기준으로 정리**한 버전입니다.

통합된 내용은 다음 3가지입니다.

1. **FAISS 저수준 인덱스 예제** (`faiss` 직접 사용)
2. **LangChain + FAISS 텍스트 유사도 검색 및 간단 Q&A**
3. **LangChain + Chroma + PDF 문서 유사도 검색**

## 주요 변경 사항

- `text-embedding-ada-002` → `text-embedding-3-small` 또는 `text-embedding-3-large`
- `gpt-3.5-turbo` → 최신 계열 모델 예시로 `gpt-4o-mini` 사용
- `langchain.document_loaders` → `langchain_community.document_loaders`
- `langchain.vectorstores` → 각 통합 패키지 사용 (`langchain_community`, `langchain_chroma`)
- `load_qa_chain(...)` 기반 예제는 최신 구조에 맞게 **검색 후 직접 프롬프트 구성** 방식으로 변경
- 하드코딩 API 키(`sk-...`) 제거, 환경변수 `OPENAI_API_KEY` 사용
- Windows 절대경로(`e:/data/...`) 대신 `pathlib.Path` 기반 경로 사용

> 실행 전, 아래 설치 셀과 환경변수 셀부터 순서대로 실행하세요.


In [ ]:
# 필요 패키지 설치
# Jupyter 환경에서 한 번만 실행하면 됩니다.
%pip install -qU numpy faiss-cpu python-dotenv langchain langchain-community langchain-openai langchain-chroma langchain-text-splitters pypdf


In [ ]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"
FAISS_INDEX_PATH = BASE_DIR / "vector.index"

DATA_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)

print(f"BASE_DIR      : {BASE_DIR}")
print(f"DATA_DIR      : {DATA_DIR}")
print(f"CHROMA_DIR    : {CHROMA_DIR}")
print(f"FAISS_INDEX   : {FAISS_INDEX_PATH}")


## 1) FAISS 기본 인덱스 예제

기존 `파이스와 인덱스.ipynb` 내용을 최신 실행 흐름으로 정리한 버전입니다.

- 임의 벡터 생성
- `IndexIVFFlat` 인덱스 생성
- 학습(train) 후 add
- 검색(search)
- 인덱스 저장 / 로드


In [ ]:
import numpy as np
import faiss

# 재현 가능성을 위해 시드 고정
np.random.seed(1)

# 데이터베이스 벡터 생성
dimension = 128
n = 200
db_vectors = np.random.random((n, dimension)).astype("float32")

print("db_vectors shape:", db_vectors.shape)


In [ ]:
nlist = 5  # IVF 클러스터 수
quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_L2)

print("is_trained(before):", index.is_trained)
print("ntotal(before)    :", index.ntotal)

index.train(db_vectors)
index.add(db_vectors)

# 검색 시 확인할 inverted list 수 (기본값보다 명시하는 편이 이해에 좋음)
index.nprobe = 2

print("is_trained(after) :", index.is_trained)
print("ntotal(after)     :", index.ntotal)
print("nprobe            :", index.nprobe)


In [ ]:
# 쿼리 벡터 생성 및 검색
n_query = 10
k = 3
query_vectors = np.random.random((n_query, dimension)).astype("float32")

distances, indices = index.search(query_vectors, k)

print("distances shape:", distances.shape)
print("indices shape  :", indices.shape)
print("첫 번째 쿼리의 top-k 인덱스:", indices[0])
print("첫 번째 쿼리의 top-k 거리  :", distances[0])


In [ ]:
# 인덱스 저장 / 로드
faiss.write_index(index, str(FAISS_INDEX_PATH))
loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))

print("saved to:", FAISS_INDEX_PATH)
print("loaded ntotal:", loaded_index.ntotal)
print("loaded is_trained:", loaded_index.is_trained)


## 2) LangChain + FAISS 텍스트 유사도 검색

기존 `파이스, 유사도 검색.ipynb`를 최신 패키지 구조에 맞게 변경한 버전입니다.

### 바뀐 점
- `langchain.text_splitter` → `langchain_text_splitters`
- `text-embedding-ada-002` → `text-embedding-3-small`
- `load_qa_chain(...)` 제거
- 검색된 문서를 직접 프롬프트에 넣어 답변 생성


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

text = """생성형 인공지능 또는 생성형 AI는 프롬프트에 대응하여 텍스트, 이미지, 기타 미디어를 생성할 수 있는 일종의 인공지능 시스템이다.
단순히 기존 데이터를 분석하는 것이 아닌, 새로운 콘텐츠를 만드는 데 초점을 맞춘 인공지능 분야를 말한다. 2022년경부터 본격적으로 유명해지기 시작했다.
데이터 원본을 통한 학습으로 소설, 이미지, 비디오, 코딩, 음악, 미술 등 다양한 콘텐츠 생성에 이용된다. 한국에서는 2022년 Novel AI 등, 그림 인공지능의 등장으로 주목도가 높아졌으며, 해외에서는 미드저니, 챗GPT 등 여러 모델을 잇달아 공개하면서 화제의 중심이 되었다.
보통 딥러닝 인공지능은 학습 혹은 결과 출력 전 원본 자료를 배열 자료형 숫자 데이터로 변환하는 인코딩 과정이 중요한데, 생성 AI의 경우 인공지능의 출력 데이터를 역으로 그림, 글 등의 원하는 형태로 변환시켜주는 디코딩 과정 또한 필요하다.
사실상 인공지능의 대중화를 이끈 기술로써, 해당 기술이 인공지능에 대한 사람들의 전반적인 인식을 매우 크게 바꿔놨다고 해도 과언이 아니다.
"""

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50,
)
chunks = splitter.split_text(text)

print("chunk 개수:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\n[chunk {i}]\n{chunk}")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_texts(chunks, embedding=embeddings)

print(vector_store)


In [ ]:
question = "생성형 AI란?"
references = vector_store.similarity_search(question, k=3)

for i, doc in enumerate(references, start=1):
    print(f"\n[reference {i}]\n{doc.page_content}")


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

context = "\n\n".join(doc.page_content for doc in references)
prompt = f"""
너는 검색된 참고 문서를 바탕으로만 답변하는 도우미다.

[질문]
{question}

[참고 문서]
{context}

[지시사항]
- 참고 문서 범위 안에서만 간결하게 답변할 것
- 한국어로 답변할 것
"""

response = llm.invoke(prompt)
print(response.content)


In [ ]:
# LangChain FAISS 벡터스토어 저장 / 로드 예시
FAISS_STORE_DIR = BASE_DIR / "faiss_store"
vector_store.save_local(str(FAISS_STORE_DIR))

loaded_vector_store = FAISS.load_local(
    str(FAISS_STORE_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)

loaded_docs = loaded_vector_store.similarity_search("생성형 인공지능의 특징은?", k=2)
print(loaded_docs[0].page_content)


## 3) LangChain + Chroma + PDF 문서 유사도 검색

기존 `크로마, 유사도 검색.ipynb`를 최신 패키지 구조에 맞게 바꾼 버전입니다.

### 바뀐 점
- `langchain.document_loaders` → `langchain_community.document_loaders`
- `langchain.vectorstores.Chroma` → `langchain_chroma.Chroma`
- `langchain.embeddings.openai.OpenAIEmbeddings` → `langchain_openai.OpenAIEmbeddings`
- 하드코딩 경로 제거
- 최신 OpenAI 임베딩 모델 사용
- `persist()` 호출 없이 `persist_directory` 기반으로 저장되도록 정리

아래 예제는 `data/` 폴더 안에 PDF 파일 2개가 있다고 가정합니다.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_paths = [
    DATA_DIR / "스마트농업_육성사업_추진현황과_개선과제.pdf",
    DATA_DIR / "차세대 한국형 스마트팜 개발.pdf",
]

missing_files = [str(path) for path in pdf_paths if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "다음 PDF 파일이 없습니다. data/ 폴더에 파일을 넣어주세요:\n- " + \"\n- \".join(missing_files)
    )

raw_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    raw_docs.extend(loader.load())

print("로드한 문서 수:", len(raw_docs))
print(raw_docs[0].metadata)


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
)

splits = text_splitter.split_documents(raw_docs)
print("청크 수:", len(splits))
print(splits[0].page_content[:500])


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="smart_farm_docs",
)

print("Chroma 저장 위치:", CHROMA_DIR)
print("collection_name  : smart_farm_docs")


In [ ]:
question = "한국형 스마트팜이란?"
docs = vectordb.similarity_search(question, k=3)

print("검색 결과 수:", len(docs))
print("\n첫 번째 결과 본문 일부:\n")
print(docs[0].page_content[:1000])
print("\n첫 번째 결과 메타데이터:\n", docs[0].metadata)


In [ ]:
question = "필요한 ICT 기술은?"
docs = vectordb.similarity_search(question, k=5)

for i, doc in enumerate(docs[:2], start=1):
    print(f"\n[result {i}] metadata = {doc.metadata}")
    print(doc.page_content[:1000])


In [ ]:
# 저장된 Chroma DB 다시 열기
reloaded_vectordb = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="smart_farm_docs",
)

reloaded_docs = reloaded_vectordb.similarity_search("스마트팜 핵심 기술", k=3)
print(reloaded_docs[0].page_content[:700])


## 정리

이 통합본에서 반영한 핵심 포인트는 아래와 같습니다.

- **FAISS 자체 사용법**은 여전히 유효하지만, `nprobe`, 저장/로드, dtype(`float32`) 등을 명확히 다루도록 정리
- **LangChain 관련 import 경로**는 예전과 많이 달라졌으므로 패키지별 import 사용
- **OpenAI 모델명**은 예전 `ada-002`, `gpt-3.5-turbo` 대신 최신 계열 예시 사용
- **Chroma**는 별도 패키지(`langchain-chroma`) 사용
- **구식 QA 체인** 대신, 검색 결과를 직접 프롬프트에 넣는 방식으로 단순하고 유지보수 쉬운 예제로 변경

필요하면 이 다음 단계로도 바로 확장할 수 있습니다.

1. 하나의 함수로 묶어서 재사용 가능한 RAG 유틸 만들기
2. PDF 여러 개를 자동 스캔해서 벡터DB 구축하기
3. Streamlit / Gradio 챗봇 UI로 연결하기
4. FAISS와 Chroma 성능 비교 실험 코드 추가하기
